# Five machine learning models on real market data

Decision trees, random forests and XGBoost on a survivorship free panel of S&P 500 stocks,
PCA on the Treasury curve, and Lasso on a wide factor panel.

Runs top to bottom. Prices come from Yahoo Finance, rates and macro from FRED, and index
membership from the Wikipedia revision history. No API key is needed anywhere.

**Expect ten to fifteen minutes on the first run**, almost all of it downloading. Two caches
are written to disk (`membership.json` and `precos.parquet`), so later runs take about three
minutes.

Full write up: https://davidariasfinance.com/scripts/machine-learning-for-finance/

## Step 0. Install and import

Everything the notebook needs, in one place, before anything runs.

In [ ]:
%pip install -q yfinance pandas numpy scikit-learn xgboost matplotlib requests lxml pyarrow

In [ ]:
import io, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import yfinance as yf

import matplotlib
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.linear_model import LassoCV, lasso_path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, log_loss, r2_score
import xgboost as xgb

START, END = "2007-01-01", "2026-08-18"
HORIZON = 5                      # trading days ahead that the classifiers predict
HEADERS = {"User-Agent": "Mozilla/5.0 (research)"}
WIKI_API = "https://en.wikipedia.org/w/api.php"

print("ready")

## Step 1. Two data helpers

FRED is read through its public CSV endpoint, which needs no key. Prices come from yfinance
with `auto_adjust=True`, so they already account for splits and dividends.

In [ ]:
def fred(series):
    """FRED through the public CSV endpoint, no API key required."""
    out = {}
    for s in series:
        r = requests.get("https://fred.stlouisfed.org/graph/fredgraph.csv",
                         params={"id": s}, timeout=90)
        r.raise_for_status()
        d = pd.read_csv(io.StringIO(r.text))
        d.columns = ["date", s]
        d["date"] = pd.to_datetime(d["date"])
        out[s] = pd.to_numeric(d.set_index("date")[s], errors="coerce")
    return pd.DataFrame(out)


def rsi(px, n=14):
    r = px.pct_change()
    gain = r.clip(lower=0).rolling(n).mean()
    loss = (-r.clip(upper=0)).rolling(n).mean()
    return 100 - 100 / (1 + gain / loss.replace(0, np.nan))

print("helpers defined")

## Step 2. A survivorship free universe

Taking today's S&P 500 list and running it back to 2007 is the most common way to produce a
flattering backtest: every company that went bankrupt or was acquired is missing, so the sample
is made only of survivors.

Wikipedia keeps a full revision history of its S&P 500 page, so the membership as it actually
stood on any past date can be read back from the revision that was live then. Snapshots every
six months give the real membership through time, delisted names included.

**Slow cell.** Forty revisions with a polite throttle. Cached to `membership.json` afterwards.

In [ ]:
CACHE = Path("membership.json")

def members_on(date):
    p = {"action": "query", "prop": "revisions", "titles": "List of S&P 500 companies",
         "rvlimit": 1, "rvstart": f"{date}T00:00:00Z", "rvdir": "older",
         "rvprop": "ids|timestamp", "format": "json", "formatversion": 2}
    for attempt in range(4):
        resp = requests.get(WIKI_API, params=p, headers=HEADERS, timeout=90)
        try:
            j = resp.json(); break
        except ValueError:                       # rate limited, back off
            time.sleep(3 * (attempt + 1))
    else:
        return []
    rev = j["query"]["pages"][0].get("revisions")
    if not rev:
        return []
    html = requests.get("https://en.wikipedia.org/w/index.php",
                        params={"oldid": rev[0]["revid"]}, headers=HEADERS, timeout=90).text
    try:
        big = [t for t in pd.read_html(io.StringIO(html)) if t.shape[0] > 300]
    except ValueError:
        return []
    if not big:
        return []
    col = [c for c in big[0].columns if str(c).lower().startswith(("symbol", "ticker"))]
    if not col:
        return []
    tick = (big[0][col[0]].astype("string").fillna("").str.strip().str.upper()
            .str.replace(".", "-", regex=False))          # BRK.B -> BRK-B, Yahoo format
    return sorted({x for x in tick.tolist()               # fillna: some revisions carry NaN
                   if isinstance(x, str) and x.isascii()
                   and 1 <= len(x) <= 6 and x.replace("-", "").isalpha()})


snapshots = json.loads(CACHE.read_text()) if CACHE.exists() else {}
wanted = [f"{y}-{m}-01" for y in range(2007, 2027) for m in ("01", "07")]
for d in [d for d in wanted if d <= END and d not in snapshots]:
    got = members_on(d)
    if got:
        snapshots[d] = got
        CACHE.write_text(json.dumps(snapshots, indent=1))
    time.sleep(1.2)

universe = sorted({t for lst in snapshots.values() for t in lst})
print(f"{len(snapshots)} snapshots, {len(universe)} tickers ever in the index")

## Step 3. Prices, and the point in time mask

Every ticker that was ever a member gets downloaded, including the ones that no longer trade.
Coverage is reported honestly: Yahoo no longer serves history for many long delisted names, so
the correction is a large improvement rather than a complete one.

Each stock then contributes rows only on the dates it was genuinely a member.

**Slow cell.** Cached to `precos.parquet` afterwards.

In [ ]:
PRICES = Path("precos.parquet")

if PRICES.exists():
    px = pd.read_parquet(PRICES)
else:
    parts = []
    for i in range(0, len(universe), 120):
        d = yf.download(universe[i:i + 120], start=START, end=END,
                        auto_adjust=True, progress=False)
        if isinstance(d.columns, pd.MultiIndex):
            d = d["Close"]
        parts.append(d)
    px = pd.concat(parts, axis=1).sort_index()
    px = px.loc[:, ~px.columns.duplicated()].dropna(axis=1, how="all")
    px.to_parquet(PRICES)

marks = sorted(snapshots)
mask = pd.DataFrame(False, index=px.index, columns=px.columns)
for i, d0 in enumerate(marks):
    d1 = marks[i + 1] if i + 1 < len(marks) else "2100-01-01"
    mask.loc[(px.index >= d0) & (px.index < d1),
             [t for t in snapshots[d0] if t in mask.columns]] = True

print(f"{px.shape[1]} of {len(universe)} tickers with data "
      f"({px.shape[1]/len(universe):.0%} coverage)")
print(f"median members on a day: {int(mask.sum(axis=1).median())}")

## Step 4. Features, and a cross sectional target

Not "will the market rise", which carries a drift a model can free ride on, but **will this
stock beat the median of its peers over the next five trading days**. That is balanced 50/50 by
construction, so the base rate sits at 0.50.

Because the question is relative, every feature also enters as its **cross sectional rank** on
the day. Raw momentum says a stock rose 3%; its rank says it rose more than 80% of its peers,
and only the second answers the question being asked.

Sampling every fifth day matters twice: the five day windows stop overlapping, so each row is
an independent observation, and the panel fits in memory.

In [ ]:
ret  = px.pct_change()
days = px.index[::HORIZON]
cols = {}

def piece(name, frame):
    q = frame.where(mask).loc[days]
    cols[name]         = q.stack().astype("float32")
    cols[f"{name}_rk"] = q.rank(axis=1, pct=True).stack().astype("float32")

for j in (1, 5, 21, 63, 126, 252):
    piece(f"mom_{j}", px / px.shift(j) - 1)               # momentum ladder
for j in (21, 63, 126):
    piece(f"vol_{j}", ret.rolling(j).std() * np.sqrt(252))  # volatility ladder
piece("dist_sma50",  px / px.rolling(50).mean()  - 1)
piece("dist_sma200", px / px.rolling(200).mean() - 1)
piece("rsi_14", rsi(px))
piece("rev_5", -(px / px.shift(5) - 1))

X = pd.concat(cols, axis=1).replace([np.inf, -np.inf], np.nan).dropna()
fwd = (px.shift(-HORIZON) / px - 1).where(mask).loc[days]
y = (fwd.rank(axis=1, pct=True) > 0.5).astype("int8").stack().reindex(X.index).dropna()
X = X.loc[y.index]

dates = X.index.get_level_values(0)
cut   = dates[int(len(X) * 0.7)]
tr, te = dates < cut, dates >= cut
Xtr, ytr, Xte, yte = X[tr], y[tr], X[te], y[te]

print(f"{len(X):,} rows x {X.shape[1]} features")
print(f"train {int(tr.sum()):,} through {cut.date()}, test {int(te.sum()):,}")
print(f"base rate on the test block: {max(yte.mean(), 1-yte.mean()):.4f}")

## Step 5. Decision tree, random forest, XGBoost

Same data, same target, three algorithms. Accuracy is reported next to **AUC**, which asks
whether the ranking is right regardless of where a 0.5 cutoff falls. On a balanced target with
a weak signal, AUC is the more informative of the two.

In [ ]:
def evaluate(name, model):
    model.fit(Xtr, ytr)
    p = model.predict_proba(Xte)[:, 1]
    acc = ((p > 0.5).astype(int) == yte).mean()
    auc = roc_auc_score(yte, p)
    # windows do not overlap, so no discount is needed on the standard error
    z = (auc - 0.5) / np.sqrt(1.0 / (12 * len(yte)))
    print(f"  {name:14} accuracy {acc:.4f}   AUC {auc:.4f}   z {z:+.1f}")
    return p

evaluate("decision tree", DecisionTreeClassifier(
    max_depth=6, min_samples_leaf=2000, random_state=0))

evaluate("random forest", RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=500, n_jobs=3, random_state=0))

p_xgb = evaluate("xgboost", xgb.XGBClassifier(
    n_estimators=700, learning_rate=0.05, max_depth=5, subsample=0.8,
    colsample_bytree=0.8, reg_lambda=2.0, min_child_weight=200,
    eval_metric="logloss", n_jobs=3))

print(f"  {'base rate':14} accuracy {max(yte.mean(), 1-yte.mean()):.4f}   AUC 0.5000")

### What the ranking is worth

Bucket the test rows by predicted probability and measure how many in each bucket went on to
beat the peer median. Signal in the tails is the normal shape, and it is why these get traded
as long and short extremes rather than as a straight classifier.

In [ ]:
d = pd.DataFrame({"p": pd.Series(p_xgb, index=Xte.index), "y": yte})
d["decile"] = pd.qcut(d["p"].rank(method="first"), 10, labels=False) + 1
rate = d.groupby("decile")["y"].mean()

fig, ax = plt.subplots(figsize=(11, 4.4))
ax.bar(rate.index, rate.values, color="#2f5db0", width=0.62)
ax.axhline(yte.mean(), color="#d6336c", ls="--", lw=1.5)
slack = max(0.012, (rate.max() - rate.min()) * 0.35)
ax.set_ylim(rate.min() - slack, rate.max() + slack)   # a zero baseline hides the whole effect
ax.set_title("Share that beat the cross sectional median, by predicted decile", loc="left")
ax.set_xlabel("decile of predicted probability, XGBoost")
ax.set_xticks(range(1, 11))
plt.show()

print(f"bottom decile {rate.iloc[0]:.4f}   top decile {rate.iloc[-1]:.4f}"
      f"   spread {rate.iloc[-1]-rate.iloc[0]:+.4f}")

## Step 6. PCA on the Treasury curve

Different problem, and the one where machine learning genuinely pays for itself here. PCA is
not a predictor at all, it is unsupervised compression.

Ten maturities move together almost all the time, so treating them as ten independent risks is
wasteful. Run it on daily **changes** rather than levels, because levels are non stationary and
the first component would simply track the drift of the whole rate era.

In [ ]:
tenors = ["DGS3MO","DGS6MO","DGS1","DGS2","DGS3","DGS5","DGS7","DGS10","DGS20","DGS30"]
curve  = fred(tenors).dropna()

pca = PCA(n_components=5).fit(curve.diff().dropna())
var = pca.explained_variance_ratio_ * 100
years = [0.25, 0.5, 1, 2, 3, 5, 7, 10, 20, 30]

fig, ax = plt.subplots(figsize=(11, 4.8))
for i, (c, lab) in enumerate(zip(["#2f5db0", "#d97706", "#0ca678"],
        [f"PC1 level {var[0]:.1f}%", f"PC2 slope {var[1]:.1f}%",
         f"PC3 curvature {var[2]:.1f}%"])):
    ax.plot(years, pca.components_[i], color=c, lw=2.2, marker="o", ms=5, label=lab)
ax.axhline(0, color="#dfe3ec", lw=1.2)
ax.set_xscale("log"); ax.set_xticks(years)
ax.set_xticklabels([f"{a:g}" for a in years])
ax.set_title("Treasury curve, loadings of the first three components", loc="left")
ax.set_xlabel("maturity in years"); ax.legend(frameon=False)
plt.show()

print(f"{len(curve.diff().dropna()):,} daily observations")
print("explained variance:", np.round(var, 2))
print(f"first three together: {var[:3].sum():.2f}%")

Nobody told the model to look for those shapes. PC1 comes out flat and positive across every
maturity, so it moves the whole curve. PC2 falls from the front end to the long end, tilting it.
PC3 is high at both ends and dips in the belly, bending it. Level, slope and curvature, recovered
from nothing but the covariance of daily changes.

One caveat: requiring all ten maturities drops February 2002 to February 2006, when the 30 year
bond was discontinued.

## Step 7. Lasso on a wide factor panel

Linear regression carrying a penalty on the size of its own coefficients. The L1 norm puts
corners on the constraint region, so coefficients land on exactly zero rather than merely
shrinking, and the model selects its variables while it fits them.

Target here is the forward 21 day return on SPY, with 22 sector and asset class ETFs plus six
macro series as candidates.

In [ ]:
etfs = ["XLK","XLF","XLE","XLV","XLI","XLY","XLP","XLU","XLB","XLRE","XLC",
        "IWM","EFA","EEM","TLT","IEF","HYG","LQD","GLD","USO","UUP","VNQ"]
bag = yf.download(etfs, start="2010-01-01", end=END, auto_adjust=True, progress=False)
bag = bag["Close"] if "Close" in bag else bag
bag = bag.dropna(axis=1, how="all").ffill().dropna()

spy = yf.download("SPY", start="2010-01-01", end=END, auto_adjust=True, progress=False)["Close"]
spy = spy.iloc[:, 0] if isinstance(spy, pd.DataFrame) else spy

mom = pd.concat({f"{c}_mom21": bag[c].pct_change(21) for c in bag.columns}, axis=1)
vol = pd.concat({f"{c}_vol21": bag[c].pct_change().rolling(21).std() for c in bag.columns}, axis=1)
macro = fred(["DGS10","DGS2","T10Y2Y","VIXCLS","BAMLH0A0HYM2","DTWEXBGS"]).reindex(bag.index).ffill()

panel = pd.concat([mom, vol, macro], axis=1).dropna()
target = (spy.shift(-21) / spy - 1).reindex(panel.index).dropna()
panel = panel.loc[target.index]

k = int(len(panel) * 0.7)
scaler = StandardScaler().fit(panel[:k])                  # fit on TRAIN only
cv = LassoCV(cv=TimeSeriesSplit(5), n_alphas=120, max_iter=20000,
             random_state=0).fit(scaler.transform(panel[:k]), target[:k])

alive = [c for c, b in zip(panel.columns, cv.coef_) if abs(b) > 1e-10]
pred  = cv.predict(scaler.transform(panel[k:]))
print(f"{len(alive)} of {panel.shape[1]} predictors survive")
print(f"survivors: {alive}")
print(f"out of sample R2 {r2_score(target[k:], pred):+.4f}   "
      f"vs {r2_score(target[k:], np.full(len(pred), target[:k].mean())):+.4f} for the mean")

In [ ]:
alphas, coefs, _ = lasso_path(scaler.transform(panel[:k]), target[:k], n_alphas=120)

fig, ax = plt.subplots(figsize=(11, 4.6))
for i in range(coefs.shape[0]):
    ax.plot(alphas, coefs[i], lw=1.3, alpha=0.9)
ax.axvline(cv.alpha_, color="#9aa1ae", ls=":", lw=1.6)
ax.axhline(0, color="#dfe3ec", lw=1.2)
ax.set_xscale("log")
ax.set_title(f"Coefficient path, {panel.shape[1]} candidate predictors", loc="left")
ax.set_xlabel("alpha")
plt.show()

## Limitations

Worth reading before trusting any number above.

**Data.** Survivorship is reduced and not eliminated: Yahoo serves usable history for roughly
three quarters of the tickers, and the missing names skew towards companies that failed, which is
exactly the group whose absence causes the bias. Membership is sampled every six months, so a
company that joined and left inside one window is invisible. Wikipedia is crowd edited and not an
index provider. Ticker reuse is unhandled. `auto_adjust=True` applies today's split and dividend
factors across the whole history. FRED series are revised, and these are current vintages.

**Method.** No transaction costs, spread, slippage or borrow anywhere, and on an edge of one point
of AUC that is the whole result rather than a rounding error. One train and test split rather than
a walk forward. Hyperparameters were chosen by hand while test results were visible, which is a
mild form of selection. One market and one era. Accuracy and AUC are not money.

These are worked examples. For work with money behind it, start from the literature:
https://davidariasfinance.com/papers/